EECE 5644 Mini Project 2 - Skylar Denno

In [14]:
############################################################

# EECE 5644 - Machine Learning
# Skylar Denno (No group currently - all work in this project is mine)
# July 2, 2026

# MINI PROJECT 2
# Data cleaning and data preprocessing

############################################################

In [15]:
import sys, subprocess
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs])

pipq("scikit-learn", "pandas", "numpy", "matplotlib", "seaborn")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)
np.random.seed(0)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [16]:
df = pd.read_csv("telco.csv", encoding="ISO-8859-1")
print("\nLoaded dataset with", df.shape[0], "rows\n\n")
display(df.describe())
#print(df.head(5))
print("\n\nNum. missing values in each column:\n", df.isnull().sum(), "n\n")


Loaded dataset with 7043 rows




,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,target,Churn,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
count,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00
mean,0.50,0.16,0.48,0.30,32.37,0.90,0.42,0.29,0.34,0.34,0.29,0.38,0.39,0.59,64.76,0.27,0.44,0.22,0.21,0.24,0.22,0.34,0.23
std,0.50,0.37,0.50,0.46,24.56,0.30,0.49,0.45,0.48,0.48,0.45,0.49,0.49,0.49,30.09,0.44,0.50,0.41,0.41,0.43,0.41,0.47,0.42
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,18.25,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00,9.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,35.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,1.00,0.00,0.00,0.00,29.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,70.35,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
75%,1.00,0.00,1.00,1.00,55.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,89.85,1.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00
max,1.00,1.00,1.00,1.00,72.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,118.75,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00




Num. missing values in each column:
 gender                                   0
SeniorCitizen                            0
Partner                                  0
Dependents                               0
tenure                                   0
PhoneService                             0
MultipleLines                            0
OnlineSecurity                           0
OnlineBackup                             0
DeviceProtection                         0
TechSupport                              0
StreamingTV                              0
StreamingMovies                          0
PaperlessBilling                         0
target                                   0
TotalCharges                             0
Churn                                    0
InternetService_Fiber optic              0
InternetService_No                       0
Contract_One year                        0
Contract_Two year                        0
PaymentMethod_Credit card (automatic)    0
PaymentMethod_E

In [17]:
print(df["PhoneService"].value_counts())
# Yes, column PhoneService contains 0s and 1s only, with no missing values.

df = df.rename(columns={
    "target" : "monthly_charge",
    "SeniorCitizen" : "senior_citizen",
    "Partner" : "partner",
    "Dependents" : "dependents",
    "PhoneService" : "phone_service",
    "MultipleLines" : "multiple_lines",
    "OnlineSecurity" : "online_security",
    "OnlineBackup" : "online_backup",
    "DeviceProtection" : "device_protection",
    "TechSupport" : "tech_support",
    "StreamingMovies" : "streaming_movies",
    "PaperlessBilling" : "paperless_billing",
    "MultipleLines" : "multiple_lines",
    "TotalCharges" : "total_charges",
    "Churn" : "churn",
    "InternetService_Fiber optic" : "fiber_optic",
    "InternetService_No" : "no_internet_service",
    "StreamingTV" : "streaming_tv"
})

PhoneService
1    6361
0     682
Name: count, dtype: int64


In [18]:

# consolidating the 3 redundant payment method columns into a single one

df["PaymentMethod_Electronic check"] = df["PaymentMethod_Electronic check"].map(lambda x: x * 2)
df["PaymentMethod_Mailed check"] = df["PaymentMethod_Mailed check"].map(lambda x: x * 3)
df["payment_method_ID"] = df["PaymentMethod_Credit card (automatic)"] + df["PaymentMethod_Electronic check"] + df["PaymentMethod_Mailed check"]
df = df.drop(columns=["PaymentMethod_Credit card (automatic)", "PaymentMethod_Electronic check", "PaymentMethod_Mailed check"])
df["payment_method_name"] = df["payment_method_ID"].map({0 : "Credit Card (Manual)", 1 : "Credit Card (Automatic)", 2 : "Electronic Check", 3 : "Mailed Check"})

df["Contract_Two year"] = df["Contract_Two year"].map(lambda x: x * 2)
df["contract_length_years"] = df["Contract_One year"] + df["Contract_Two year"]
df = df.drop(columns=["Contract_One year", "Contract_Two year"])

df["gender"] = df["gender"].map({0: "Female", 1: "Male"})

df.info()
print(df["payment_method_ID"].value_counts())
print(df["payment_method_name"].value_counts())
print(df["contract_length_years"].value_counts())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   gender                 7043 non-null   object 
 1   senior_citizen         7043 non-null   int64  
 2   partner                7043 non-null   int64  
 3   dependents             7043 non-null   int64  
 4   tenure                 7043 non-null   int64  
 5   phone_service          7043 non-null   int64  
 6   multiple_lines         7043 non-null   int64  
 7   online_security        7043 non-null   int64  
 8   online_backup          7043 non-null   int64  
 9   device_protection      7043 non-null   int64  
 10  tech_support           7043 non-null   int64  
 11  streaming_tv           7043 non-null   int64  
 12  streaming_movies       7043 non-null   int64  
 13  paperless_billing      7043 non-null   int64  
 14  monthly_charge         7043 non-null   float64
 15  tota

In [ ]:
# feature columns, target = monthly_charge
feature_cols = ["phone_service", "multiple_lines", "online_security", "online_backup",
                "device_protection", "tech_support", "streaming_tv", "streaming_movies",
                "fiber_optic", "no_internet_service"]

X = df[feature_cols]
y = df["monthly_charge"]

# form training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=777)

# fit multivariable lin regression on traning set
telcomodel = LinearRegression()
telcomodel.fit(X_train, y_train)

# Extract learned parameters
omega_0 = telcomodel.intercept_
omegas = pd.Series(telcomodel.coef_, index=feature_cols, name="omega_i")

print("Learned intercept omega_0:", round(omega_0, 4))
print("\nLearned coefficients omega_i:")
display(omegas.to_frame())

# Predict on the test set
y_pred = telcomodel.predict(X_test)

# Test results
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R2  :", r2)
print("MAE : $", mae)
print("RMSE: $", rmse)

results = pd.DataFrame()
results["actual"] = y_test.values
results["predicted"] = y_pred.round(2)
results["error"] = results["actual"] - results["predicted"]
results["error_sq"] = (results["actual"] - results["predicted"]) ** 2
print(results.head(10))

Learned intercept omega_0: 24.9369

Learned coefficients omega_i:


,omega_i
phone_service,20.07
multiple_lines,5.00
online_security,5.02
online_backup,5.01
device_protection,5.00
tech_support,5.05
streaming_tv,9.98
streaming_movies,9.97
fiber_optic,24.95
no_internet_service,-25.05


R2  : 0.9988533669069787
MAE : $ 0.7757835407720778
RMSE: $ 1.0291697275218272
   actual  predicted  error  error_sq
0   20.05      19.96   0.09      0.01
1   58.65      59.97  -1.32      1.74
2   21.05      19.96   1.09      1.19
3   58.20      60.08  -1.88      3.53
4  101.10      99.91   1.19      1.42
5   54.35      55.08  -0.73      0.53
6   88.75      89.94  -1.19      1.42
7   79.15      79.94  -0.79      0.62
8   55.70      55.00   0.70      0.49
9   74.65      75.01  -0.36      0.13
